# Q-Mini Light 0.7M — One-Click GPU English Assistant

**एक ही मुख्य Code Cell है।** नवीनतम सुधार 16 Nov 2026 को प्रकाशित है। पुराना खुला Colab tab बंद करके नीचे दिए लिंक को दोबारा खोलें, फिर Runtime → Run all एक बार दबाएँ। Notebook क्रम से डेटा तैयार करेगा, GPU preflight करेगा, ट्रेनिंग या resume करेगा, हर checkpoint साझा Google Drive में सत्यापित करके सेव करेगा, आपकी किताबों की searchable library बनाएगा और अंत में Q-Mini chat शुरू करेगा।

## पहली बार की जरूरी Drive व्यवस्था

1. जिस नए Google account से Colab चलाना है, उसी account को आपके QMini-Light-0.7M shared folder में **Editor** रखें।
2. नए account में Google Drive → **Shared with me** खोलें। QMini-Light-0.7M पर right-click → **Organize → Add shortcut** → **My Drive** चुनें।
3. Colab खोलें। यदि folder My Drive में नहीं है, Notebook आपके दिए folder-link ID से उसी नए account के My Drive में shortcut बनाने की कोशिश करेगा। इसके लिए account को Editor access होना जरूरी है।

Shared link अकेले upload permission नहीं देता। फ़ोल्डर में Editor access और Colab Drive authorization, दोनों जरूरी हैं। पुराना account mounted हो तो Notebook remount करेगा; फिर shared folder ID से shortcut बनाने की कोशिश करेगा। सुरक्षा के लिए बेहतर है General access को Restricted रखें और केवल नए account को Editor बनाएँ; Anyone with link + Editor रखने पर link पाने वाला कोई भी व्यक्ति फ़ाइलें बदल सकता है।

यदि पुरानी CPU Notebook अभी training कर रही है, उसे पहले Runtime → Interrupt execution से रोकें। एक ही model/checkpoint store पर दो training sessions साथ न चलाएँ।

## यह Notebook क्या करेगा

- Q-Mini के मौजूदा matching 0.7M / 512-context model, config और tokenizer को जाँचकर इस्तेमाल करेगा; नया random model नहीं बनाएगा।
- English-only multi-turn conversation dataset में WildChat-4.8M के non-toxic public split से अधिकतम 25,000 साफ़ संवाद और उपलब्ध/नया बनाया गया OpenAssistant OASST1 human-reviewed English subset मिलाएगा।
- कोई Mistral/API key, paid API या paid dataset नहीं। WildChat का gated Full dataset नहीं लिया जाएगा।
- पहले से तैयार मानवीय OASST1 JSONL उपलब्ध हो तो उसे दोबारा डाउनलोड नहीं करेगा।
- Assistant-only loss mask, tokenizer-manifest-matched held-out validation, duplicate conversation-tree reduction और assistant turn-ending token की जाँच करेगा।
- उपलब्ध CUDA GPU पर train करेगा। GPU/VRAM कम हो तो training शुरू करने से पहले साफ़ error देगा; CPU पर चुपचाप लंबी training नहीं करेगा।
- हर 25 steps पर checkpoint, metrics और run manifest को Drive पर immutable generation के रूप में upload करेगा; SHA-256/CRC verify होने के बाद ही progress counter update होगा। पिछली दो verified generations सुरक्षित रहेंगी।
- पहले के फ़ोल्डर/मॉडल नहीं हटाएगा। नया run अपनी अलग folder में बनेगा।
- Drive में रखी TXT, Markdown, CSV, JSON/JSONL, PDF, DOCX और Parquet पुस्तकों/दस्तावेज़ों से अलग local searchable knowledge library बनाएगा। किताबें मॉडल weights में fine-tune नहीं होंगी; chat में relevant अंश दिए जाएँगे।
- अंत में उसी trained model से interactive English chat चलेगी। exit लिखकर chat बंद कर सकते हैं।

**ईमानदार सीमा:** 0.7M छोटा research model है। अच्छे संवादों से सुधार हो सकता है, लेकिन इससे ChatGPT-जितनी क्षमता की गारंटी नहीं होती। इस Notebook की code/structure जाँच की गई है; वास्तविक GPU training और आपके दूसरे Google account का Drive access केवल आपके Colab runtime में चलाने पर ही प्रमाणित होगा.

[Google Drive folder](https://drive.google.com/drive/folders/1WS2KW12zCRRy6re3noZ4TE1uuwFzXlQXaaiae) · [GitHub source](https://github.com/lion36508/Q-mini-light-0.7M)

In [ ]:
# ================================================================
# Q-MINI ONE-CELL GPU TRAIN + VERIFIED DRIVE CHECKPOINTS + RAG CHAT
# NOTEBOOK PATCH: 2026-11-16-syntax-fix-02
# Press Runtime -> Run all once. Do not run another Q-Mini trainer
# in parallel against the same Drive checkpoint folder.
# ================================================================

import os, sys, json, re, csv, time, math, shutil, hashlib, random
import subprocess, importlib.util, sqlite3, textwrap, uuid
from pathlib import Path
from datetime import datetime, timezone

# ---- 1) Mount Drive; locate/create a shortcut to the shared project folder ----
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
MY_DRIVE = DRIVE_MOUNT / "MyDrive"
PROJECT_DRIVE_NAME = "QMini-Light-0.7M"
SHARED_FOLDER_ID = "1WS2KW12zCRRy6re3noZ4TE1uuwFzXlQXaaiae"
PROJECT_DRIVE = None

# First reuse an existing mount. If folder is absent, remount once so the
# user can select the NEW Google account, not the old training account.
drive.mount("/content/drive", force_remount=False)

def find_project_folder():
    direct = MY_DRIVE / PROJECT_DRIVE_NAME
    if direct.is_dir():
        return direct
    shared_drives = DRIVE_MOUNT / "Shareddrives"
    if shared_drives.is_dir():
        for root, dirs, files in os.walk(shared_drives):
            dirs[:] = [d for d in dirs if not d.startswith(".")]
            if PROJECT_DRIVE_NAME in dirs:
                return Path(root) / PROJECT_DRIVE_NAME
            if len(Path(root).relative_to(shared_drives).parts) >= 3:
                dirs[:] = []
    return None

PROJECT_DRIVE = find_project_folder()
if PROJECT_DRIVE is None:
    print("The project folder is not in the mounted account's My Drive. Remounting once...")
    drive.mount("/content/drive", force_remount=True)
    MY_DRIVE = DRIVE_MOUNT / "MyDrive"
    PROJECT_DRIVE = find_project_folder()

# If the shared folder is still not mounted, use its exact shared-link ID
# to create a normal Drive shortcut in this account's My Drive.
# This does not copy, move, or alter the shared folder's existing contents.
if PROJECT_DRIVE is None:
    try:
        from google.colab import auth
        auth.authenticate_user()
        import google.auth
        from googleapiclient.discovery import build

        credentials, _ = google.auth.default(
            scopes=["https://www.googleapis.com/auth/drive"]
        )
        drive_api = build(
            "drive", "v3", credentials=credentials, cache_discovery=False
        )
        target = drive_api.files().get(
            fileId=SHARED_FOLDER_ID,
            supportsAllDrives=True,
            fields="id,name,mimeType,capabilities(canEdit)"
        ).execute()

        if target.get("mimeType") != "application/vnd.google-apps.folder":
            raise ValueError(
                "The supplied shared link ID does not identify a Google Drive folder."
            )
        if target.get("capabilities", {}).get("canEdit") is not True:
            raise PermissionError(
                "The new Google account can open the shared folder but does not have Editor "
                "permission. Add this exact account as Editor, then run this cell again."
            )

        query = (
            "trashed=false and mimeType='application/vnd.google-apps.shortcut' "
            f"and shortcutDetails.targetId='{SHARED_FOLDER_ID}'"
        )
        shortcuts = drive_api.files().list(
            q=query, pageSize=100, spaces="drive",
            fields="files(id,name,shortcutDetails)"
        ).execute().get("files", [])

        if not shortcuts:
            drive_api.files().create(
                body={
                    "name": PROJECT_DRIVE_NAME,
                    "mimeType": "application/vnd.google-apps.shortcut",
                    "shortcutDetails": {"targetId": SHARED_FOLDER_ID},
                    "parents": ["root"],
                },
                supportsAllDrives=True,
                fields="id,name,mimeType,shortcutDetails"
            ).execute()
            print("Created a shortcut to the shared Q-Mini folder in this account's My Drive.")
        else:
            print("A shortcut to the shared Q-Mini folder already exists in this account.")

        # Refresh the mounted filesystem so that the newly-created shortcut appears.
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        drive.mount("/content/drive", force_remount=True)
        MY_DRIVE = DRIVE_MOUNT / "MyDrive"
        PROJECT_DRIVE = find_project_folder()

    except Exception as exc:
        raise RuntimeError(
            "Could not create/find the shared Q-Mini folder shortcut using the supplied link. "
            "This usually means the link ID is incorrect, the new account is not shared as "
            "Editor, or Drive did not grant API access. Open the shared folder link in the "
            "NEW account and verify its General access/Editor permission. "
            f"Details: {type(exc).__name__}: {str(exc)[:500]}"
        ) from exc

if PROJECT_DRIVE is None:
    try:
        visible_items = sorted(p.name for p in MY_DRIVE.iterdir())[:40]
    except Exception:
        visible_items = []
    raise FileNotFoundError(
        "The shortcut was created or access was checked, but the folder has not appeared in "
        "the mounted My Drive yet. Restart the Colab runtime and run this same cell once. "
        f"Visible My Drive items: {visible_items}"
    )

print("Google Drive project folder found:", PROJECT_DRIVE)

# Verify Editor/write access BEFORE downloading datasets or starting GPU work.
# The temporary probe is created and removed immediately; no existing file is touched.
write_probe = PROJECT_DRIVE / f".qmini_write_test_{uuid.uuid4().hex}.tmp"
try:
    write_probe.write_text("Q-Mini Drive write permission test\\n", encoding="utf-8")
    if write_probe.read_text(encoding="utf-8") != "Q-Mini Drive write permission test\\n":
        raise IOError("Google Drive write probe did not round-trip correctly.")
finally:
    write_probe.unlink(missing_ok=True)
print("Google Drive read/write permission: PASS")

# ---- 2) Install lightweight dependencies without replacing Colab's Torch ----
def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", pip_name or import_name],
            check=True
        )

for imp, pkg in [
    ("datasets", "datasets>=3.0"),
    ("tokenizers", "tokenizers>=0.15"),
    ("pyarrow", "pyarrow>=15"),
    ("pypdf", "pypdf"),
    ("docx", "python-docx"),
    ("packaging", "packaging>=23"),
]:
    ensure_package(imp, pkg)

import torch
from packaging.version import Version
if Version(torch.__version__.split("+", 1)[0]) < Version("2.10.0"):
    raise RuntimeError(
        f"Q-Mini needs PyTorch >=2.10.0; Colab currently has {torch.__version__}. "
        "Use a current Colab GPU runtime. This Notebook will not replace your CUDA Torch wheel."
    )
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available. Select Runtime -> Change runtime type -> T4 GPU "
        "(or another supported GPU), save, and run this same cell again. CPU training is disabled."
    )

DEVICE = "cuda"
GPU_NAME = torch.cuda.get_device_name(0)
FREE_BYTES, TOTAL_BYTES = torch.cuda.mem_get_info(0)
FREE_GIB, TOTAL_GIB = FREE_BYTES / 1024**3, TOTAL_BYTES / 1024**3
print(f"GPU: {GPU_NAME} | free VRAM: {FREE_GIB:.2f} / {TOTAL_GIB:.2f} GiB")
if FREE_GIB < 8.0:
    raise RuntimeError(
        f"Only {FREE_GIB:.2f} GiB GPU memory is free; this run requires at least 8 GiB free. "
        "Close other GPU sessions or reconnect to a roomier GPU before training."
    )

torch.manual_seed(42)
random.seed(42)
try:
    torch.set_num_threads(4)
except Exception:
    pass

# ---- 3) Get the exact repository training/runtime source ----
REPO_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"
REPO_DIR = Path("/content/Q-mini-light-0.7M")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True
    )
else:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", "main"],
        cwd=REPO_DIR, check=True
    )
sys.path.insert(0, str(REPO_DIR))

from qmini.config import QMiniConfig
from qmini.model import QMiniMoE
from qmini.tokenizer import load_tokenizer
from qmini.text import SPECIAL_TOKENS, escape_special_token_literals
from qmini.data import prepare_token_streams
from qmini.export import export_packed_state, verify_packed_export
from qmini.train import train_model, _verify_initial_model_artifact
from qmini.checkpointing import (
    TrainingSessionLease, restore_latest_checkpoint, mark_training_status
)
from qmini.generate import load_model, generate

# ---- 4) Locate the verified existing 0.7M model + matching tokenizer ----
PREFERRED_PACKAGE = (
    PROJECT_DRIVE / "training_ready"
    / "QMini-English-512-Q2v2-e1a187f0ca1567cd"
)
if PREFERRED_PACKAGE.is_dir():
    BASE_PACKAGE = PREFERRED_PACKAGE
else:
    package_candidates = sorted(
        [p for p in (PROJECT_DRIVE / "training_ready").glob("QMini-English-512-Q2v2-*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )
    if not package_candidates:
        raise FileNotFoundError(
            "No QMini-English-512-Q2v2 package found under "
            "QMini-Light-0.7M/training_ready. Nothing has been modified."
        )
    BASE_PACKAGE = package_candidates[0]

checkpoint_candidates = list((BASE_PACKAGE / "training_state").glob("checkpoint-step-*.pt"))
if not checkpoint_candidates:
    raise FileNotFoundError(
        f"No saved training checkpoint in {BASE_PACKAGE / 'training_state'}. "
        "This Notebook will not start from random weights."
    )

def checkpoint_step(p):
    match = re.search(r"checkpoint-step-(\d+)\.pt$", p.name)
    return int(match.group(1)) if match else -1

checkpoint_candidates.sort(key=checkpoint_step, reverse=True)
CONFIG_CANDIDATES = [
    BASE_PACKAGE / "active_config.json",
    BASE_PACKAGE / "initial_brain" / "config.json",
    BASE_PACKAGE / "config.json",
]
CONFIG_PATH = next((p for p in CONFIG_CANDIDATES if p.is_file()), None)
TOKENIZER_PATH = BASE_PACKAGE / "tokenizer" / "tokenizer.json"
if CONFIG_PATH is None or not TOKENIZER_PATH.is_file():
    raise FileNotFoundError("Matching Q-Mini config/tokenizer not found beside the selected package.")

CONFIG = QMiniConfig.from_json(CONFIG_PATH)
TOKENIZER = load_tokenizer(TOKENIZER_PATH)
if CONFIG.estimated_parameter_count() != 705742 or CONFIG.max_context != 512:
    raise ValueError("Selected package is not the expected Q-Mini Light 0.7M / context-512 profile.")
if TOKENIZER.get_vocab_size() != CONFIG.vocab_size:
    raise ValueError("Tokenizer vocabulary size does not match the model configuration.")
for special_token in SPECIAL_TOKENS:
    if TOKENIZER.token_to_id(special_token) is None:
        raise ValueError(f"Matching tokenizer is missing required token {special_token!r}.")

BASE_CHECKPOINT = None
BASE_STATE = None
for candidate in checkpoint_candidates:
    try:
        state = torch.load(candidate, map_location="cpu", weights_only=True)
        if not isinstance(state, dict) or state.get("format") != "qmini-checkpoint-v2":
            continue
        if int(state.get("step", 0)) < 1:
            continue
        if QMiniConfig.from_dict(state.get("config")).to_dict() != CONFIG.to_dict():
            continue
        if not isinstance(state.get("model"), dict):
            continue
        BASE_CHECKPOINT, BASE_STATE = candidate, state
        break
    except Exception as exc:
        print("Skipping invalid checkpoint:", candidate.name, type(exc).__name__)

if BASE_CHECKPOINT is None:
    raise RuntimeError("No compatible non-zero-step Q-Mini training checkpoint passed verification.")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

BASE_SHA = sha256_file(BASE_CHECKPOINT)
TOKENIZER_SHA = sha256_file(TOKENIZER_PATH)
print("Existing model:", BASE_CHECKPOINT)
print("Base optimizer step:", int(BASE_STATE["step"]))
print("Model parameters:", f"{CONFIG.estimated_parameter_count():,}")
print("Tokenizer SHA:", TOKENIZER_SHA[:16])

# ---- 5) Build/reuse English human-AI conversation data ----
# WildChat is streamed; the complete 3.2M dataset is NOT downloaded.
# Only English, non-toxic, non-redacted user/assistant messages are retained.
DATA_DIR = PROJECT_DRIVE / "datasets" / "english_conversation"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = DATA_DIR / "qmini_english_human_ai_conversations_v1.jsonl"
DATASET_META = DATA_DIR / "qmini_english_human_ai_conversations_v1.manifest.json"

SYSTEM_PROMPT = (
    "You are Q-Mini, a friendly, clear and useful English AI assistant. "
    "Answer the user's actual question directly. Be accurate, explain reasoning when useful, "
    "admit uncertainty instead of inventing facts, and keep multi-turn context."
)

WILDCHAT_LIMIT = 25000
OASST_OLD_PATH = (
    PROJECT_DRIVE / "assistant_sft_cpu_runs"
    / "assistant_sft_oasst1_human_english_v1"
    / "qmini_oasst1_human_english.jsonl"
)

def valid_chat_messages(raw_messages, max_chars=6000):
    cleaned = []
    for item in raw_messages or []:
        if not isinstance(item, dict):
            continue
        role = str(item.get("role") or item.get("author") or item.get("speaker") or "").lower()
        if isinstance(item.get("author"), dict):
            role = str(item["author"].get("role") or role).lower()
        if role in ("prompter", "human"):
            role = "user"
        if role not in ("user", "assistant"):
            continue
        body = item.get("content", item.get("text", ""))
        if not isinstance(body, str):
            continue
        body = re.sub(r"\s+", " ", body).strip()
        if len(body) < 2:
            continue
        if item.get("toxic") is True or item.get("redacted") is True:
            continue
        body = body[:max_chars]
        if cleaned and cleaned[-1]["role"] == role:
            cleaned[-1]["content"] = (cleaned[-1]["content"] + "\n" + body)[:max_chars]
        else:
            cleaned.append({"role": role, "content": body})
    while cleaned and cleaned[0]["role"] != "user":
        cleaned.pop(0)
    while cleaned and cleaned[-1]["role"] != "assistant":
        cleaned.pop()
    if len(cleaned) < 2:
        return None
    cleaned = cleaned[-6:]
    while cleaned and cleaned[0]["role"] != "user":
        cleaned.pop(0)
    if len(cleaned) < 2 or cleaned[-1]["role"] != "assistant":
        return None
    return [{"role": "system", "content": SYSTEM_PROMPT}] + cleaned

def load_existing_oasst(path):
    records = []
    if not path.is_file():
        return records
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            messages = row.get("messages", [])
            if not isinstance(messages, list):
                continue
            normalized = valid_chat_messages(messages)
            if normalized is None:
                continue
            meta = row.get("metadata", {})
            if (
                meta.get("source") != "OpenAssistant/oasst1"
                or meta.get("synthetic") is not False
                or str(meta.get("language", "")).lower() not in ("en", "english")
            ):
                continue
            records.append({
                "messages": normalized,
                "metadata": {
                    "source": "OpenAssistant/oasst1",
                    "license": "Apache-2.0",
                    "synthetic": False,
                    "language": "en",
                    "seed_group": str(meta.get("seed_group") or uuid.uuid4().hex),
                }
            })
    # Keep only the longest assistant-complete path per conversation tree.
    # This prevents shared conversation prefixes leaking across train/validation.
    best_by_tree = {}
    for record in records:
        tree = record["metadata"]["seed_group"]
        prior = best_by_tree.get(tree)
        if prior is None or len(record["messages"]) > len(prior["messages"]):
            best_by_tree[tree] = record
    return list(best_by_tree.values())

def build_oasst_from_public():
    from datasets import load_dataset
    ds = load_dataset("OpenAssistant/oasst1")
    by_id = {}
    for split_name in ds.keys():
        for row in ds[split_name]:
            if row.get("lang") != "en":
                continue
            if row.get("tree_state") != "ready_for_export":
                continue
            if row.get("review_result") is not True or row.get("deleted") is not False:
                continue
            if row.get("synthetic") is not False:
                continue
            if row.get("role") not in ("prompter", "assistant"):
                continue
            body = re.sub(r"\s+", " ", str(row.get("text") or "")).strip()
            if body:
                item = dict(row)
                item["text"] = body
                by_id[item["message_id"]] = item

    result, seen = [], set()
    for assistant in by_id.values():
        if assistant.get("role") != "assistant":
            continue
        if assistant.get("rank") not in (None, 0):
            continue
        message_id = assistant["message_id"]
        if message_id in seen:
            continue
        chain, visited, current = [], set(), assistant
        complete = True
        while current is not None:
            mid = current.get("message_id")
            if mid in visited:
                complete = False
                break
            visited.add(mid)
            chain.append(current)
            parent_id = current.get("parent_id")
            if not parent_id:
                break
            current = by_id.get(parent_id)
            if current is None:
                complete = False
                break
        if not complete:
            continue
        chain.reverse()
        tree_id = assistant.get("message_tree_id")
        if not tree_id or len(chain) < 2:
            continue
        if chain[0].get("role") != "prompter" or chain[-1].get("role") != "assistant":
            continue
        expected = ["prompter" if i % 2 == 0 else "assistant" for i in range(len(chain))]
        if [m["role"] for m in chain] != expected:
            continue
        if any(m.get("message_tree_id") != tree_id for m in chain):
            continue
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for msg in chain[-6:]:
            messages.append({
                "role": "user" if msg["role"] == "prompter" else "assistant",
                "content": msg["text"][:6000],
            })
        result.append({
            "messages": messages,
            "metadata": {
                "source": "OpenAssistant/oasst1",
                "license": "Apache-2.0",
                "synthetic": False,
                "language": "en",
                "seed_group": str(tree_id),
            }
        })
        seen.add(message_id)
    best_by_tree = {}
    for record in result:
        tree = record["metadata"]["seed_group"]
        prior = best_by_tree.get(tree)
        if prior is None or len(record["messages"]) > len(prior["messages"]):
            best_by_tree[tree] = record
    return list(best_by_tree.values())

if DATASET_PATH.exists() != DATASET_META.exists():
    raise RuntimeError(
        "Conversation dataset and its manifest are not both present. "
        "The existing file is being preserved; inspect the folder before trying a new version."
    )
if DATASET_PATH.is_file() and DATASET_META.is_file():
    old_meta = json.loads(DATASET_META.read_text(encoding="utf-8"))
    if old_meta.get("system_prompt_version") == "qmini-english-assistant-v1":
        print("Reusing saved conversation dataset:", DATASET_PATH)
    else:
        raise RuntimeError("Existing conversation data manifest differs; refusing to overwrite it.")
else:
    print("Preparing conversation dataset. Public dataset downloads are free; no API keys are used.")
    oasst_records = load_existing_oasst(OASST_OLD_PATH)
    if not oasst_records:
        print("Saved OASST1 JSONL not found; fetching public OpenAssistant/oasst1.")
        oasst_records = build_oasst_from_public()
    if len(oasst_records) < 1000:
        raise RuntimeError(f"Only {len(oasst_records)} valid OASST1 conversations found; stopping safely.")

    from datasets import load_dataset
    wild_records = []
    wild_source = "allenai/WildChat-4.8M"
    try:
        wild_stream = load_dataset(wild_source, split="train", streaming=True)
    except Exception as first_error:
        print("WildChat-4.8M stream unavailable; trying the public 650K WildChat release.")
        wild_source = "allenai/WildChat"
        try:
            wild_stream = load_dataset(wild_source, split="train", streaming=True)
        except Exception as second_error:
            raise RuntimeError(
                "Could not open either public WildChat stream. Check Colab Internet access. "
                f"First error: {type(first_error).__name__}; second: {type(second_error).__name__}"
            )

    if hasattr(wild_stream, "shuffle"):
        wild_stream = wild_stream.shuffle(seed=42, buffer_size=10000)

    scanned = 0
    for row in wild_stream:
        scanned += 1
        if scanned > 600000:
            break
        if str(row.get("language", "")).strip().lower() not in ("english", "en"):
            continue
        if row.get("toxic") is not False or row.get("redacted") is not False:
            continue
        messages = valid_chat_messages(row.get("conversation", []))
        if messages is None:
            continue
        per_message_languages = [
            str(m.get("language", "")).strip().lower()
            for m in row.get("conversation", [])
            if isinstance(m, dict) and m.get("language")
        ]
        if per_message_languages and any(x not in ("english", "en") for x in per_message_languages):
            continue
        conversation_hash = row.get("conversation_hash")
        if not isinstance(conversation_hash, str) or not conversation_hash.strip():
            conversation_hash = uuid.uuid4().hex

        wild_records.append({
            "messages": messages,
            "metadata": {
                "source": wild_source,
                "license": "ODC-BY",
                "synthetic": False,
                "language": "en",
                "seed_group": conversation_hash,
            }
        })
        if len(wild_records) >= WILDCHAT_LIMIT:
            break
        if len(wild_records) % 2500 == 0:
            print(f"Accepted {len(wild_records):,} WildChat English conversations (scanned {scanned:,})")

    if len(wild_records) < 1000:
        raise RuntimeError(
            f"Only {len(wild_records)} English WildChat records survived the quality filters. "
            "Refusing to start a tiny or poorly parsed training set."
        )

    all_records = oasst_records + wild_records
    random.Random(42).shuffle(all_records)
    temporary_dataset = DATASET_PATH.with_suffix(".jsonl.tmp")
    with temporary_dataset.open("w", encoding="utf-8") as handle:
        for item in all_records:
            handle.write(json.dumps(item, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary_dataset, DATASET_PATH)
    meta = {
        "format": "qmini-english-chat-data-v1",
        "system_prompt_version": "qmini-english-assistant-v1",
        "wildchat_source": wild_source,
        "wildchat_license": "ODC-BY",
        "wildchat_examples": len(wild_records),
        "oasst_source": "OpenAssistant/oasst1",
        "oasst_license": "Apache-2.0",
        "oasst_examples": len(oasst_records),
        "total_examples": len(all_records),
        "synthetic_examples": 0,
        "language": "English only",
        "privacy": "Conversation messages only; IP, country, state and request headers excluded.",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    temporary_meta = DATASET_META.with_suffix(".json.tmp")
    temporary_meta.write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")
    os.replace(temporary_meta, DATASET_META)
    print("New English conversation dataset saved:", DATASET_PATH)
    print(json.dumps(meta, indent=2))

record_count = 0
bad_count = 0
with DATASET_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue
        record_count += 1
        try:
            row = json.loads(line)
            if row.get("metadata", {}).get("synthetic") is not False:
                bad_count += 1
            roles = [m.get("role") for m in row.get("messages", [])]
            if "user" not in roles or "assistant" not in roles or roles[-1] != "assistant":
                bad_count += 1
        except Exception:
            bad_count += 1
if record_count < 2000 or bad_count:
    raise RuntimeError(f"Dataset validation failed: records={record_count}, invalid={bad_count}.")
DATA_SHA = sha256_file(DATASET_PATH)
print(f"Validated {record_count:,} English chat records | SHA-256 {DATA_SHA[:16]}")

# ---- 6) Create a verified Q2-v2 base model folder for official SFT trainer ----
RUN_NAME = (
    "qmini_english_wildchat_oasst1_"
    + DATA_SHA[:10] + "_" + BASE_SHA[:8] + "_" + TOKENIZER_SHA[:8]
)
RUN_DIR = PROJECT_DRIVE / "assistant_sft_gpu_runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
BASE_MODEL_DIR = RUN_DIR / "base_model"
BASE_Q2_PATH = BASE_MODEL_DIR / "model.qmini.npz"

if not (BASE_MODEL_DIR / "model_manifest.json").is_file():
    base_model = QMiniMoE(CONFIG)
    incompatible = base_model.load_state_dict(BASE_STATE["model"], strict=True)
    if incompatible.missing_keys or incompatible.unexpected_keys:
        raise RuntimeError("Existing checkpoint tensors did not load strictly.")
    BASE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    (BASE_MODEL_DIR / "config.json").write_text(
        json.dumps(CONFIG.to_dict(), indent=2) + "\n", encoding="utf-8"
    )
    shutil.copy2(TOKENIZER_PATH, BASE_MODEL_DIR / "tokenizer.json")
    export_packed_state(base_model.state_dict(), CONFIG, BASE_Q2_PATH)
    packed_report = verify_packed_export(BASE_Q2_PATH)
    if packed_report.get("format") != "qmini-packed-v2" or packed_report.get("q2_version") != 2:
        raise RuntimeError("Existing model could not be converted into a verified Q2-v2 training seed.")
    packed_sha = sha256_file(BASE_Q2_PATH)
    seed_manifest = {
        "format": "qmini-final-model-v2",
        "step": int(BASE_STATE["step"]),
        "tokens_seen": int(BASE_STATE.get("tokens_seen", 0)),
        "supervised_tokens_seen": int(BASE_STATE.get("supervised_tokens_seen", 0)),
        "tokenizer_sha256": TOKENIZER_SHA,
        "tokenizer_size_bytes": TOKENIZER_PATH.stat().st_size,
        "parameters": CONFIG.estimated_parameter_count(),
        "matrix_weight_format": "qmini-packed-v2",
        "matrix_weight_bits": 2,
        "q2_codebook": {"00": -1.0, "01": -0.5, "10": 0.5, "11": 1.0},
        "packed_export_verification": packed_report,
        "packed_export": BASE_Q2_PATH.name,
        "packed_export_bytes": BASE_Q2_PATH.stat().st_size,
        "packed_export_sha256": packed_sha,
        "trainable_parameter_dtypes": ["torch.float32"],
        "trainable_weight_policy": "fp32-master-q2-ste",
        "optimizer_state_dtypes": [],
        "dense_snapshot_sha256": None,
        "dense_snapshot_bytes": None,
        "estimated_q2_fp16_raw_bytes": CONFIG.estimated_packed_export_bytes(),
        "deployment_weight_budget_bytes": 1500000000,
        "within_1_5gb_weight_artifact_budget": True,
        "dense_training_snapshot_included": False,
        "resume_checkpoint": str(BASE_CHECKPOINT),
        "config": CONFIG.to_dict(),
    }
    if seed_manifest["tokens_seen"] < 1:
        raise RuntimeError("Base checkpoint is missing tokens_seen; refusing an unverified SFT start.")
    (BASE_MODEL_DIR / "model_manifest.json").write_text(
        json.dumps(seed_manifest, indent=2) + "\n", encoding="utf-8"
    )
    print("Created immutable Q2-v2 seed from existing trained checkpoint.")
else:
    print("Reusing previously verified Q2-v2 seed model.")
_verify_initial_model_artifact(BASE_Q2_PATH, CONFIG, TOKENIZER_PATH)
print("Verified SFT seed:", BASE_Q2_PATH)

# ---- 7) Prepare deduplicated train/validation streams with assistant-only loss ----
PREPARED_DIR = RUN_DIR / "prepared_dataset"
TOKENIZER_SIDECAR = Path(str(TOKENIZER_PATH) + ".qmini.json")
if TOKENIZER_SIDECAR.is_file():
    tokenizer_split_meta = json.loads(TOKENIZER_SIDECAR.read_text(encoding="utf-8"))
    VALIDATION_FRACTION = float(tokenizer_split_meta.get("validation_fraction", 0.02))
    NEAR_DUPLICATE_DISTANCE = int(tokenizer_split_meta.get("near_duplicate_hamming_distance", 0))
else:
    VALIDATION_FRACTION = 0.02
    NEAR_DUPLICATE_DISTANCE = 0
print("Validation split / near-duplicate policy matches the saved tokenizer:", VALIDATION_FRACTION, NEAR_DUPLICATE_DISTANCE)
if not (PREPARED_DIR / "dataset.json").is_file():
    PREPARED_DIR.mkdir(parents=True, exist_ok=True)
    dataset_report = prepare_token_streams(
        [DATASET_PATH],
        TOKENIZER_PATH,
        PREPARED_DIR,
        valid_fraction=VALIDATION_FRACTION,
        loss_mask_mode="assistant",
        near_duplicate_distance=NEAR_DUPLICATE_DISTANCE,
    )
    print("Prepared train/validation streams:")
    print(json.dumps(dataset_report, indent=2))
else:
    print("Reusing saved prepared dataset:", PREPARED_DIR)

TRAIN_BIN = PREPARED_DIR / "train.bin"
VALID_BIN = PREPARED_DIR / "valid.bin"
TRAIN_MASK = PREPARED_DIR / "train.mask.bin"
VALID_MASK = PREPARED_DIR / "valid.mask.bin"
for required_path in (TRAIN_BIN, VALID_BIN, TRAIN_MASK, VALID_MASK):
    if not required_path.is_file() or required_path.stat().st_size == 0:
        raise FileNotFoundError(f"Prepared Q-Mini training file missing/empty: {required_path}")
if TRAIN_BIN.stat().st_size // 2 != TRAIN_MASK.stat().st_size:
    raise ValueError("Training loss-mask length does not match training tokens.")
if VALID_BIN.stat().st_size // 2 != VALID_MASK.stat().st_size:
    raise ValueError("Validation loss-mask length does not match validation tokens.")
print(
    "Prepared token counts:",
    f"train={TRAIN_BIN.stat().st_size // 2:,}",
    f"validation={VALID_BIN.stat().st_size // 2:,}",
    "| context=512 | loss=assistant-only"
)

# ---- 8) GPU fine-tuning, immutable Drive checkpoints and safe resume ----
TARGET_STEPS = 2000
BATCH_SIZE = 2
GRAD_ACCUM = 4
SEQUENCE_LENGTH = CONFIG.max_context
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.05
EVALUATE_EVERY = 50
CHECKPOINT_EVERY = 25
EVALUATION_BATCHES = 8
MIRROR_DIR = RUN_DIR / "training_state"
LOCAL_RUN_DIR = Path("/content/qmini_gpu_local_runs") / RUN_NAME
MIRROR_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

with TrainingSessionLease(
    MIRROR_DIR, stale_after_seconds=180, heartbeat_seconds=45
) as lease:
    progress_path = MIRROR_DIR / "training_progress.json"
    current_progress = (
        json.loads(progress_path.read_text(encoding="utf-8"))
        if progress_path.is_file() else None
    )
    restored = restore_latest_checkpoint(MIRROR_DIR, LOCAL_RUN_DIR)

    if restored is None:
        if current_progress and int(current_progress.get("current_step", 0)) > 0:
            raise RuntimeError(
                "Drive progress is above step 0 but no verified checkpoint could be restored. "
                "The original files are preserved; refusing to restart from the beginning."
            )
        resume_run = False
        extend_schedule = False
        if current_progress is None:
            progress_path.write_text(json.dumps({
                "format": "qmini-training-progress-v1",
                "status": "not_started",
                "current_step": 0,
                "target_steps": TARGET_STEPS,
                "progress_percent": 0.0,
                "tokens_seen": 0,
                "updated_at": datetime.now(timezone.utc).isoformat(),
                "note": "Initial state. No optimizer update has run yet."
            }, indent=2) + "\n", encoding="utf-8")
        start_step = 0
    else:
        resume_run = True
        start_step = int(restored["step"])
        previous_target = int(restored["target_steps"])
        if previous_target > TARGET_STEPS and start_step < TARGET_STEPS:
            raise ValueError(
                f"Previous run target is {previous_target}; do not lower it to {TARGET_STEPS}."
            )
        TARGET_STEPS = max(TARGET_STEPS, start_step)
        extend_schedule = TARGET_STEPS > previous_target
        print("Resuming from verified Drive step:", start_step)

    if current_progress and current_progress.get("status") == "running":
        mark_training_status(
            MIRROR_DIR, "interrupted",
            note="Previous Colab process ended; newest verified Drive checkpoint will be restored."
        )
    mark_training_status(
        MIRROR_DIR, "running",
        note=f"GPU training session started on {GPU_NAME} at {datetime.now(timezone.utc).isoformat()}."
    )

    FINAL_LOCAL_DIR = train_model(
        config=CONFIG,
        train_bin=TRAIN_BIN,
        valid_bin=VALID_BIN,
        output_dir=LOCAL_RUN_DIR,
        steps=TARGET_STEPS,
        batch_size=BATCH_SIZE,
        seq_len=SEQUENCE_LENGTH,
        grad_accum=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        eval_every=EVALUATE_EVERY,
        save_every=CHECKPOINT_EVERY,
        eval_batches=EVALUATION_BATCHES,
        resume=resume_run,
        tokenizer_path=TOKENIZER_PATH,
        seed=42,
        extend_schedule=extend_schedule,
        train_mask_bin=TRAIN_MASK,
        valid_mask_bin=VALID_MASK,
        init_from=None if resume_run else BASE_Q2_PATH,
        checkpoint_mirror_dir=MIRROR_DIR,
        checkpoint_session_id=lease.session_id,
    )

    FINAL_MANIFEST_PATH = FINAL_LOCAL_DIR / "model_manifest.json"
    FINAL_MANIFEST = json.loads(FINAL_MANIFEST_PATH.read_text(encoding="utf-8"))
    FINAL_PACKED_PATH = FINAL_LOCAL_DIR / FINAL_MANIFEST["packed_export"]
    FINAL_REPORT = verify_packed_export(FINAL_PACKED_PATH)
    FINAL_LOCAL_TOKENIZER = FINAL_LOCAL_DIR / "tokenizer.json"
    if (
        not FINAL_LOCAL_TOKENIZER.is_file()
        or sha256_file(FINAL_LOCAL_TOKENIZER) != FINAL_MANIFEST.get("tokenizer_sha256")
    ):
        raise RuntimeError("Final model tokenizer is missing or does not match the final-model manifest.")
    if (
        FINAL_MANIFEST.get("format") != "qmini-final-model-v2"
        or FINAL_REPORT.get("format") != "qmini-packed-v2"
        or FINAL_REPORT.get("q2_version") != 2
        or FINAL_REPORT.get("parameter_count") != CONFIG.estimated_parameter_count()
        or sha256_file(FINAL_PACKED_PATH) != FINAL_MANIFEST.get("packed_export_sha256")
    ):
        raise RuntimeError("Final Q-Mini Q2-v2 model failed its manifest/archive verification.")

    FINAL_STEP = int(FINAL_MANIFEST["step"])
    TRAINED_ROOT = PROJECT_DRIVE / "trained_models"
    TRAINED_ROOT.mkdir(parents=True, exist_ok=True)
    FINAL_NAME = f"{RUN_NAME}-step-{FINAL_STEP:012d}"
    FINAL_DRIVE_DIR = TRAINED_ROOT / FINAL_NAME
    STAGING_DIR = TRAINED_ROOT / f".{FINAL_NAME}.staging-{uuid.uuid4().hex[:8]}"

    if FINAL_DRIVE_DIR.exists():
        existing = json.loads((FINAL_DRIVE_DIR / "model_manifest.json").read_text(encoding="utf-8"))
        existing_npz = FINAL_DRIVE_DIR / existing["packed_export"]
        existing_tokenizer = FINAL_DRIVE_DIR / "tokenizer.json"
        if (
            existing.get("packed_export_sha256") != FINAL_MANIFEST.get("packed_export_sha256")
            or sha256_file(existing_npz) != existing.get("packed_export_sha256")
            or not existing_tokenizer.is_file()
            or sha256_file(existing_tokenizer) != existing.get("tokenizer_sha256")
        ):
            raise FileExistsError("A different or incomplete model already exists at the final Drive destination; no overwrite performed.")
        verify_packed_export(existing_npz)
    else:
        try:
            shutil.copytree(FINAL_LOCAL_DIR, STAGING_DIR)
            copied_manifest = json.loads((STAGING_DIR / "model_manifest.json").read_text(encoding="utf-8"))
            copied_npz = STAGING_DIR / copied_manifest["packed_export"]
            copied_tokenizer = STAGING_DIR / "tokenizer.json"
            if (
                sha256_file(copied_npz) != copied_manifest.get("packed_export_sha256")
                or not copied_tokenizer.is_file()
                or sha256_file(copied_tokenizer) != copied_manifest.get("tokenizer_sha256")
            ):
                raise IOError("Drive model upload failed packed-weight/tokenizer SHA-256 verification.")
            verify_packed_export(copied_npz)
            STAGING_DIR.rename(FINAL_DRIVE_DIR)
        except Exception:
            shutil.rmtree(STAGING_DIR, ignore_errors=True)
            raise

    directory_hash = hashlib.sha256()
    for file_path in sorted(FINAL_DRIVE_DIR.rglob("*")):
        if file_path.is_file():
            directory_hash.update(file_path.relative_to(FINAL_DRIVE_DIR).as_posix().encode("utf-8"))
            directory_hash.update(sha256_file(file_path).encode("ascii"))
    final_model_sha = directory_hash.hexdigest()
    completed_progress = mark_training_status(
        MIRROR_DIR,
        "completed",
        final_model_sha256=final_model_sha,
        note=f"Verified final Q2-v2 model stored at {FINAL_DRIVE_DIR}"
    )

print("\nTRAINING / RESUME COMPLETE")
print("GPU:", GPU_NAME)
print("Optimizer step:", completed_progress["current_step"], "/", completed_progress["target_steps"])
print("Drive progress:", MIRROR_DIR / "training_progress.json")
print("Verified recovery checkpoints:", MIRROR_DIR)
print("Final trained model:", FINAL_DRIVE_DIR)

# ---- 9) Build a separate searchable library from books/documents in Drive ----
KNOWLEDGE_DIR = PROJECT_DRIVE / "local_knowledge_library"
KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)
KNOWLEDGE_DB = KNOWLEDGE_DIR / (
    "qmini_knowledge_" + RUN_NAME + "_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".sqlite3"
)

SKIP_DIRS = {
    ".git", ".cache", "__pycache__", "training_ready", "trained_models",
    "assistant_sft_gpu_runs", "assistant_sft_cpu_runs", "training_state",
    "checkpoints", "node_modules"
}
SUPPORTED = {".txt", ".text", ".md", ".markdown", ".csv", ".json", ".jsonl", ".ndjson", ".pdf", ".docx", ".parquet"}

def extract_json_text(obj, out, key=""):
    blocked = {"hashed_ip", "ip", "country", "state", "header", "request_headers", "email", "phone"}
    if key.lower() in blocked:
        return
    if isinstance(obj, dict):
        for k, v in obj.items():
            extract_json_text(v, out, str(k))
    elif isinstance(obj, list):
        for v in obj:
            extract_json_text(v, out, key)
    elif isinstance(obj, str) and len(obj.strip()) > 20:
        out.append(obj.strip())

def read_document(path):
    suffix = path.suffix.lower()
    try:
        if suffix in {".txt", ".text", ".md", ".markdown"}:
            return path.read_text(encoding="utf-8", errors="ignore")[:2_000_000]
        if suffix == ".csv":
            rows = []
            with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
                for i, row in enumerate(csv.reader(f)):
                    if i >= 20000:
                        break
                    rows.append(" | ".join(row))
            return "\n".join(rows)
        if suffix in {".json", ".jsonl", ".ndjson"}:
            out = []
            with path.open("r", encoding="utf-8", errors="ignore") as f:
                if suffix == ".json":
                    extract_json_text(json.load(f), out)
                else:
                    for i, line in enumerate(f):
                        if i >= 20000:
                            break
                        if line.strip():
                            try:
                                extract_json_text(json.loads(line), out)
                            except json.JSONDecodeError:
                                out.append(line.strip())
            return "\n".join(out)[:2_000_000]
        if suffix == ".pdf":
            from pypdf import PdfReader
            reader = PdfReader(str(path))
            return "\n".join((page.extract_text() or "") for page in reader.pages[:500])[:2_000_000]
        if suffix == ".docx":
            from docx import Document
            doc = Document(str(path))
            pieces = [p.text for p in doc.paragraphs]
            for table in doc.tables:
                for row in table.rows:
                    pieces.append(" | ".join(cell.text for cell in row.cells))
            return "\n".join(pieces)[:2_000_000]
        if suffix == ".parquet":
            import pyarrow.parquet as pq
            parquet = pq.ParquetFile(path)
            pieces = []
            total = 0
            for batch in parquet.iter_batches(batch_size=256):
                data = batch.to_pydict()
                for i in range(batch.num_rows):
                    row = {key: values[i] for key, values in data.items()}
                    row_parts = []
                    extract_json_text(row, row_parts)
                    pieces.extend(row_parts)
                    total += sum(len(x) for x in row_parts)
                    if total > 2_000_000:
                        return "\n".join(pieces)[:2_000_000]
            return "\n".join(pieces)[:2_000_000]
    except Exception as exc:
        print("Knowledge-file skipped:", path.name, type(exc).__name__, str(exc)[:120])
    return ""

# Create a new versioned index. Never delete/overwrite an earlier knowledge DB.
connection = sqlite3.connect(KNOWLEDGE_DB)
connection.execute("CREATE VIRTUAL TABLE chunks USING fts5(source UNINDEXED, body, tokenize='unicode61')")
indexed_files, indexed_chunks = 0, 0
for root, dirs, files in os.walk(PROJECT_DRIVE):
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS and not d.startswith(".")]
    for filename in files:
        path_obj = Path(root) / filename
        if path_obj.suffix.lower() not in SUPPORTED:
            continue
        if path_obj == DATASET_PATH or path_obj == DATASET_META:
            continue
        try:
            size = path_obj.stat().st_size
            if size <= 0 or size > 512 * 1024 * 1024:
                continue
            document_text = read_document(path_obj)
            if not document_text.strip():
                continue
            chunks = []
            start = 0
            while start < len(document_text):
                piece = document_text[start:start + 1600].strip()
                if len(piece) >= 80:
                    chunks.append(piece)
                start += 1350
            connection.executemany(
                "INSERT INTO chunks(source, body) VALUES (?, ?)",
                [(str(path_obj.relative_to(PROJECT_DRIVE)), piece) for piece in chunks]
            )
            indexed_files += 1
            indexed_chunks += len(chunks)
            if indexed_files % 25 == 0:
                connection.commit()
                print(f"Knowledge library: {indexed_files} files / {indexed_chunks} chunks")
        except Exception as exc:
            print("File skipped:", path_obj.name, type(exc).__name__, str(exc)[:120])
connection.commit()
connection.close()
print(f"Knowledge library ready: {indexed_files} files, {indexed_chunks} searchable chunks.")
print("SQLite index:", KNOWLEDGE_DB)

def search_knowledge(query, limit=3):
    terms = re.findall(r"[A-Za-z0-9_]{2,}", query.lower())[:10]
    if not terms:
        return []
    match_query = " OR ".join('"' + t.replace('"', '""') + '"' for t in terms)
    con = sqlite3.connect(KNOWLEDGE_DB)
    try:
        rows = con.execute(
            "SELECT source, body FROM chunks WHERE chunks MATCH ? ORDER BY bm25(chunks) LIMIT ?",
            (match_query, limit)
        ).fetchall()
    finally:
        con.close()
    return rows

# ---- 10) Start interactive chat with the verified trained model ----
CHAT_MODEL, CHAT_TOKENIZER, CHAT_DEVICE = load_model(str(FINAL_DRIVE_DIR), device=DEVICE)
CHAT_MODEL.eval()
chat_history = []
CHAT_SYSTEM = (
    SYSTEM_PROMPT
    + " Use the user's book library excerpts when relevant. If the excerpts do not answer the "
      "question, say so; do not pretend that an uncited book passage supports the answer."
)

print("\n" + "=" * 66)
print("Q-MINI ENGLISH CHAT IS READY")
print("Model:", FINAL_DRIVE_DIR.name)
print("Inference device:", CHAT_DEVICE)
print("Knowledge sources indexed:", indexed_files)
print("Type 'exit' to stop; type 'clear' to clear the conversation.")
print("=" * 66)

while True:
    try:
        question = input("\nYou: ").strip()
    except EOFError:
        print("\nChat input closed.")
        break
    if not question:
        continue
    if question.lower() in {"exit", "quit", "stop"}:
        print("Q-Mini chat ended.")
        break
    if question.lower() == "clear":
        chat_history.clear()
        print("Conversation history cleared.")
        continue

    retrieved = search_knowledge(question, limit=3)
    reference_text = ""
    if retrieved:
        reference_text = "\n\nRelevant passages from the user's Drive library:\n"
        for idx, (source, body) in enumerate(retrieved, 1):
            reference_text += f"\n[{idx}] Source: {source}\n{body[:900]}\n"
    system_text = escape_special_token_literals(CHAT_SYSTEM + reference_text)
    safe_question = escape_special_token_literals(question)
    history = list(chat_history[-6:])

    def make_chat_prompt(hist, sys_text, user_text):
        prompt = f"<|system|>{sys_text}<|end|>"
        for role, content in hist:
            prompt += f"<|{role}|>{escape_special_token_literals(content)}<|end|>"
        prompt += f"<|user|>{user_text}<|end|><|assistant|>"
        return prompt

    prompt = make_chat_prompt(history, system_text, safe_question)
    while len(CHAT_TOKENIZER.encode(prompt).ids) > CONFIG.max_context - 96 and history:
        history = history[2:] if len(history) >= 2 else []
        prompt = make_chat_prompt(history, system_text, safe_question)
    if len(CHAT_TOKENIZER.encode(prompt).ids) > CONFIG.max_context - 96 and retrieved:
        reference_text = "\n\nBook excerpts:\n" + "\n".join(body[:300] for _, body in retrieved[:2])
        system_text = escape_special_token_literals(CHAT_SYSTEM + reference_text)
        prompt = make_chat_prompt(history, system_text, safe_question)
    if len(CHAT_TOKENIZER.encode(prompt).ids) > CONFIG.max_context - 96:
        safe_question = safe_question[-1200:]
        prompt = make_chat_prompt([], escape_special_token_literals(CHAT_SYSTEM), safe_question)

    try:
        answer = generate(
            CHAT_MODEL,
            CHAT_TOKENIZER,
            prompt,
            max_new_tokens=96,
            temperature=0.7,
            top_k=40,
            top_p=0.9,
            thinking_steps=CONFIG.thinking_steps,
            serialized_prompt=True,
            frequency_penalty=0.20,
            presence_penalty=0.08,
        )
    except Exception as exc:
        print("Generation error:", type(exc).__name__, str(exc)[:300])
        continue

    answer = (answer or "").strip()
    print("\nQ-Mini:", answer if answer else "[No text generated]")
    if retrieved:
        print("\nBook sources:")
        for source, _ in retrieved:
            print(" -", source)
    if answer:
        chat_history.append(("user", question))
        chat_history.append(("assistant", answer))

# ---- 11) One-line finish report ----
print("\nQ-MINI SESSION REPORT")
print("Training progress:", MIRROR_DIR / "training_progress.json")
print("Latest verified checkpoints:", MIRROR_DIR)
print("Final model:", FINAL_DRIVE_DIR)
print("Conversation dataset:", DATASET_PATH)
print("Knowledge DB:", KNOWLEDGE_DB)
print("All pre-existing Drive files were left in place.")